# MySQL Lakeflow Connect - Incremental Ingestion Setup

This notebook sets up **incremental ingestion** from MySQL (claims_dev database) to Unity Catalog using Databricks **Lakeflow Connect**.

## Architecture

1. **MySQL Source**: Database with binlog enabled for CDC
2. **Gateway Pipeline**: Extracts snapshot and change data from MySQL binlog → Unity Catalog staging volume
3. **Ingestion Pipeline**: Applies changes from staging → destination streaming tables

## Tables to Ingest
- `policy` - Insurance policy information
- `claim` - Claims data with relationship to policies
- `customer` - Customer demographics

## Prerequisites ✓
- MySQL binlog enabled (ROW format)
- MySQL user with REPLICATION SLAVE, REPLICATION CLIENT privileges
- Unity Catalog enabled
- Serverless compute enabled
- CREATE CONNECTION privileges on metastore

## Configuration
All settings are configured via **query parameters** at the top of this notebook. Update them before running.

In [0]:
# Install Databricks SDK for programmatic connection and pipeline management
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import ConnectionType
import json
import time

# Initialize Databricks workspace client
w = WorkspaceClient()

print("✓ Libraries imported successfully")
print(f"✓ Connected to workspace: {w.config.host}")

In [0]:
# Get configuration values from query parameters
config = {
    'mysql_host': dbutils.widgets.get('mysql_host'),
    'mysql_port': dbutils.widgets.get('mysql_port'),
    'mysql_database': dbutils.widgets.get('mysql_database'),
    'mysql_user': dbutils.widgets.get('mysql_user'),
    'mysql_password': dbutils.widgets.get('mysql_password'),
    'target_catalog': dbutils.widgets.get('target_catalog'),
    'target_schema': dbutils.widgets.get('target_schema'),
    'connection_name': dbutils.widgets.get('connection_name'),
    'gateway_name': dbutils.widgets.get('gateway_name'),
    'ingestion_pipeline_name': dbutils.widgets.get('ingestion_pipeline_name')
}

# Display configuration (mask password)
print("Configuration loaded:")
for key, value in config.items():
    if 'password' in key:
        print(f"  {key}: {'*' * 8}")
    else:
        print(f"  {key}: {value}")

print("\n✓ Configuration validated")

In [0]:
# Create Unity Catalog connection to MySQL
# This stores credentials securely in Unity Catalog

try:
    # Check if connection already exists
    existing_connection = w.connections.get(config['connection_name'])
    print(f"✓ Connection '{config['connection_name']}' already exists")
    print(f"  Connection ID: {existing_connection.connection_id}")
    print(f"  Connection Type: {existing_connection.connection_type.value}")
    connection_id = existing_connection.connection_id
    
except Exception as e:
    if "does not exist" in str(e) or "not found" in str(e).lower():
        print(f"Creating new connection '{config['connection_name']}'...")
        
        # Create new connection
        connection = w.connections.create(
            name=config['connection_name'],
            connection_type=ConnectionType.MYSQL,
            options={
                'host': config['mysql_host'],
                'port': config['mysql_port'],
                'user': config['mysql_user'],
                'password': config['mysql_password']
            },
            comment=f"MySQL connection for {config['mysql_database']} incremental ingestion"
        )
        
        print(f"✓ Connection created successfully")
        print(f"  Connection ID: {connection.connection_id}")
        print(f"  Connection Name: {connection.name}")
        print(f"  Connection Type: {connection.connection_type.value}")
        connection_id = connection.connection_id
    else:
        print(f"❌ Error: {e}")
        raise

In [0]:
# Ensure target catalog and schema exist

try:
    # Check if catalog exists
    catalog_info = spark.sql(f"DESCRIBE CATALOG {config['target_catalog']}").collect()
    print(f"✓ Catalog '{config['target_catalog']}' exists")
except:
    print(f"Creating catalog '{config['target_catalog']}'...")
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {config['target_catalog']}")
    print(f"✓ Catalog created")

try:
    # Check if schema exists
    schema_info = spark.sql(f"DESCRIBE SCHEMA {config['target_catalog']}.{config['target_schema']}").collect()
    print(f"✓ Schema '{config['target_catalog']}.{config['target_schema']}' exists")
except:
    print(f"Creating schema '{config['target_catalog']}.{config['target_schema']}'...")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['target_catalog']}.{config['target_schema']}")
    print(f"✓ Schema created")

print(f"\n✓ Target location ready: {config['target_catalog']}.{config['target_schema']}")

## Step 1: Create Gateway Pipeline

The **Gateway Pipeline** is a continuous DLT pipeline that:
* Connects to MySQL and extracts data from the binlog
* Performs initial snapshot load of existing data
* Continuously monitors for changes (INSERT, UPDATE, DELETE)
* Stores raw CDC data in Unity Catalog staging volumes

**Note:** The gateway must run continuously to maintain binlog position and avoid data loss due to binlog rotation.

In [0]:
# Define the gateway pipeline configuration
# This pipeline extracts data from MySQL binlog to staging volumes

gateway_config = {
    "name": config['gateway_name'],
    "storage": f"/pipelines/{config['gateway_name']}",
    "configuration": {
        "source": "mysql",
        "connection_name": config['connection_name'],
        "mysql_database": config['mysql_database'],
        "staging_catalog": config['target_catalog'],
        "staging_schema": config['target_schema']
    },
    "continuous": True,  # Gateway must run continuously
    "target": f"{config['target_catalog']}.{config['target_schema']}",
    "libraries": [],
    "clusters": [{
        "label": "default",
        "num_workers": 1,
        "custom_tags": {
            "usage": "mysql_cdc_gateway"
        }
    }]
}

print("Gateway Pipeline Configuration:")
print(json.dumps(gateway_config, indent=2))
print("\n✓ Gateway configuration ready")

In [0]:
# Create the gateway pipeline using Databricks SDK
# Note: For MySQL connector (Preview), you may need to use Databricks CLI or UI
# This code shows the programmatic approach

try:
    # Check if gateway already exists
    existing_pipelines = w.pipelines.list_pipelines(filter=f"name LIKE '{config['gateway_name']}'")
    pipeline_exists = False
    
    for pipeline in existing_pipelines:
        if pipeline.name == config['gateway_name']:
            print(f"✓ Gateway pipeline '{config['gateway_name']}' already exists")
            print(f"  Pipeline ID: {pipeline.pipeline_id}")
            print(f"  State: {pipeline.state}")
            gateway_pipeline_id = pipeline.pipeline_id
            pipeline_exists = True
            break
    
    if not pipeline_exists:
        print(f"\nNote: MySQL Lakeflow Connect is in Preview.")
        print(f"Gateway pipeline creation may require using:")
        print(f"  1. Databricks UI: Data Ingestion > MySQL > Create Pipeline")
        print(f"  2. Databricks CLI with appropriate connector configuration")
        print(f"\nFor now, please create the gateway pipeline manually via UI.")
        print(f"Configuration to use:")
        print(f"  - Gateway Name: {config['gateway_name']}")
        print(f"  - Connection: {config['connection_name']}")
        print(f"  - Staging: {config['target_catalog']}.{config['target_schema']}")
        print(f"  - Source Database: {config['mysql_database']}")
        
        gateway_pipeline_id = None  # Will be set manually
        
except Exception as e:
    print(f"❌ Error checking gateway: {e}")
    gateway_pipeline_id = None

## Step 2: Create Ingestion Pipeline with SQL

The **Ingestion Pipeline** is a DLT pipeline that:
* Reads CDC data from the staging volume (written by gateway)
* Applies changes to destination streaming tables
* Supports SCD Type 1 (latest) and Type 2 (history tracking)
* Can run on-demand or scheduled

We'll create the pipeline using SQL with streaming tables.

In [0]:
# Define the tables from claims_dev database to ingest
tables_to_ingest = [
    {
        "source_table": "policy",
        "target_table": "policy",
        "primary_key": "policy_no",
        "description": "Insurance policy information",
        "scd_type": 1  # Type 1: Only keep latest version
    },
    {
        "source_table": "claim",
        "target_table": "claim",
        "primary_key": "claim_no",
        "description": "Claims data with policy relationships",
        "scd_type": 1  # Type 1: Only keep latest version
    },
    {
        "source_table": "customer",
        "target_table": "customer",
        "primary_key": "customer_id",
        "description": "Customer demographic information",
        "scd_type": 2  # Type 2: Keep full history with timestamps
    }
]

print("Tables configured for ingestion:\n")
for table in tables_to_ingest:
    scd_label = "Latest only" if table['scd_type'] == 1 else "Full history"
    print(f"  ✓ {table['source_table']} → {config['target_catalog']}.{config['target_schema']}.{table['target_table']}")
    print(f"    Primary Key: {table['primary_key']}, SCD Type: {table['scd_type']} ({scd_label})")
    print()

In [0]:
# Create SQL notebook for the ingestion pipeline
# This notebook will define streaming tables that consume from staging

ingestion_sql = f"""
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # MySQL Claims Ingestion Pipeline
-- MAGIC 
-- MAGIC This pipeline ingests data from MySQL staging volumes into Unity Catalog streaming tables.
-- MAGIC Source: claims_dev database
-- MAGIC Target: {config['target_catalog']}.{config['target_schema']}

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Policy Table (SCD Type 1)

-- COMMAND ----------

CREATE OR REFRESH STREAMING TABLE policy
COMMENT "Insurance policy information - incrementally loaded from MySQL"
AS SELECT 
  policy_no,
  cust_id,
  policytype,
  pol_issue_date,
  pol_eff_date,
  pol_expiry_date,
  make,
  model,
  model_year,
  chassis_no,
  use_of_vehicle,
  product,
  sum_insured,
  premium,
  deductable
FROM STREAM read_files(
  's3://your-staging-volume-path/policy',  -- Update with actual staging path
  format => 'parquet'
);

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Claim Table (SCD Type 1)

-- COMMAND ----------

CREATE OR REFRESH STREAMING TABLE claim
COMMENT "Claims data - incrementally loaded from MySQL"
AS SELECT 
  claim_no,
  policy_no,
  claim_date,
  months_as_customer,
  injury,
  property,
  vehicle,
  total,
  collision_type,
  number_of_vehicles_involved,
  driver_age,
  insured_relationship,
  license_issue_date,
  incident_date,
  incident_hour,
  incident_type,
  incident_severity,
  number_of_witnesses,
  suspicious_activity
FROM STREAM read_files(
  's3://your-staging-volume-path/claim',  -- Update with actual staging path
  format => 'parquet'
);

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Customer Table (SCD Type 2 - History Tracking)

-- COMMAND ----------

CREATE OR REFRESH STREAMING TABLE customer
COMMENT "Customer information with full history tracking - incrementally loaded from MySQL"
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
)
AS SELECT 
  customer_id,
  date_of_birth,
  borough,
  neighborhood,
  zip_code,
  name,
  current_timestamp() as _loaded_at,
  cast(null as timestamp) as _valid_from,
  cast(null as timestamp) as _valid_to
FROM STREAM read_files(
  's3://your-staging-volume-path/customer',  -- Update with actual staging path
  format => 'parquet'
);
"""

print("Ingestion Pipeline SQL:")
print("=" * 80)
print(ingestion_sql[:500] + "...")
print("\n✓ SQL notebook content prepared")
print("\nNote: The actual pipeline creation via UI will:")
print("  1. Use the gateway's staging volume automatically")
print("  2. Handle CDC operations (INSERT, UPDATE, DELETE)")
print("  3. Apply SCD type 1 or 2 based on your configuration")

## Creating Pipelines via Databricks UI

Since MySQL Lakeflow Connect is in **Preview**, follow these steps in the Databricks UI:

### Step 1: Create Gateway Pipeline
1. Go to **Data Ingestion** in the sidebar
2. Click **MySQL** under Databricks connectors
3. Configure the **Ingestion Gateway**:
   * **Name**: `claims_mysql_gateway`
   * **Staging Catalog**: `main`
   * **Staging Schema**: `claims_mysql`
4. Click **Next**

### Step 2: Configure Ingestion Pipeline
5. Configure the **Ingestion Pipeline**:
   * **Name**: `claims_mysql_ingestion`
   * **Connection**: Select `mysql_claims_connection`
   * **Destination Catalog**: `main`
   * **Destination Schema**: `claims_mysql`
6. Click **Create pipeline and continue**

### Step 3: Select Tables
7. On the **Source** page:
   * Select database: `claims_dev`
   * Select tables: `policy`, `claim`, `customer`
   * For `customer` table, enable **History Tracking (SCD Type 2)**
8. Click **Next**

### Step 4: Finalize and Run
9. Review destination settings
10. Optionally set a refresh schedule
11. Click **Save and run pipeline**

### Step 5: Monitor
Both gateway and ingestion pipelines will start. Monitor their progress in the pipeline view.

In [0]:
# Helper function to check pipeline status
# Run this after creating pipelines via UI to monitor their progress

def check_pipeline_status(pipeline_name):
    """Check the status of a pipeline by name"""
    try:
        pipelines = w.pipelines.list_pipelines(filter=f"name LIKE '{pipeline_name}'")
        
        for pipeline in pipelines:
            if pipeline.name == pipeline_name:
                print(f"\nPipeline: {pipeline.name}")
                print(f"  ID: {pipeline.pipeline_id}")
                print(f"  State: {pipeline.state}")
                print(f"  URL: {w.config.host}/pipelines/{pipeline.pipeline_id}")
                
                # Get latest update
                try:
                    updates = w.pipelines.list_updates(
                        pipeline_id=pipeline.pipeline_id,
                        max_results=1
                    )
                    for update in updates:
                        print(f"\n  Latest Update:")
                        print(f"    Update ID: {update.update_id}")
                        print(f"    State: {update.state}")
                        if hasattr(update, 'creation_time'):
                            print(f"    Created: {update.creation_time}")
                except Exception as e:
                    print(f"  Could not fetch updates: {e}")
                
                return pipeline.pipeline_id
        
        print(f"Pipeline '{pipeline_name}' not found")
        return None
        
    except Exception as e:
        print(f"Error checking pipeline: {e}")
        return None

print("✓ Pipeline status checker ready")
print("\nUsage:")
print(f"  gateway_id = check_pipeline_status('{config['gateway_name']}')")
print(f"  ingestion_id = check_pipeline_status('{config['ingestion_pipeline_name']}')")

In [0]:
# Check if gateway pipeline was created and get its status
gateway_pipeline_id = check_pipeline_status(config['gateway_name'])

In [0]:
# Check if ingestion pipeline was created and get its status
ingestion_pipeline_id = check_pipeline_status(config['ingestion_pipeline_name'])

## Step 3: Verify Data Ingestion

Once the pipelines are running, verify that data is being ingested correctly:
* Check table existence
* View sample data
* Validate row counts
* Check data freshness

In [0]:
# Check if the tables have been created and get basic info

target_location = f"{config['target_catalog']}.{config['target_schema']}"

print(f"Checking tables in {target_location}...\n")

try:
    tables_df = spark.sql(f"SHOW TABLES IN {target_location}")
    tables = tables_df.collect()
    
    if len(tables) == 0:
        print("⚠️  No tables found yet. Pipelines may still be running initial load.")
        print("\nCheck pipeline status above and wait for them to complete.")
    else:
        print(f"✓ Found {len(tables)} tables:\n")
        display(tables_df)
        
        # Get details for each expected table
        for table_config in tables_to_ingest:
            table_name = table_config['target_table']
            full_table_name = f"{target_location}.{table_name}"
            
            try:
                # Get row count
                count_df = spark.sql(f"SELECT COUNT(*) as row_count FROM {full_table_name}")
                row_count = count_df.collect()[0]['row_count']
                
                print(f"\n✓ Table: {table_name}")
                print(f"  Rows: {row_count:,}")
                print(f"  Primary Key: {table_config['primary_key']}")
                print(f"  SCD Type: {table_config['scd_type']}")
                
            except Exception as e:
                print(f"\n⚠️  Table {table_name}: {e}")
                
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure the pipelines have completed their initial run.")

In [0]:
%sql
-- View sample data from policy table
SELECT 
  policy_no,
  cust_id,
  policytype,
  pol_issue_date,
  pol_eff_date,
  make,
  model,
  model_year,
  sum_insured,
  premium
FROM main.claims_mysql.policy
LIMIT 10

In [0]:
%sql
-- View sample data from claim table
SELECT 
  claim_no,
  policy_no,
  claim_date,
  total,
  collision_type,
  incident_type,
  incident_severity,
  suspicious_activity
FROM main.claims_mysql.claim
LIMIT 10

In [0]:
%sql
-- View sample data from customer table
-- This table has SCD Type 2, so it may include historical records
SELECT 
  customer_id,
  name,
  date_of_birth,
  borough,
  neighborhood,
  zip_code
FROM main.claims_mysql.customer
LIMIT 10

In [0]:
%sql
-- Check data relationships between tables
-- Join claims with policies to verify foreign key relationships
SELECT 
  c.claim_no,
  c.policy_no,
  c.total as claim_amount,
  c.incident_type,
  p.policytype,
  p.make,
  p.model,
  p.sum_insured,
  p.cust_id
FROM main.claims_mysql.claim c
INNER JOIN main.claims_mysql.policy p ON c.policy_no = p.policy_no
LIMIT 20

In [0]:
# Check when data was last updated
# This helps verify that incremental ingestion is working

import datetime

print("Data Freshness Check\n" + "=" * 60)

for table_config in tables_to_ingest:
    table_name = table_config['target_table']
    full_table_name = f"{config['target_catalog']}.{config['target_schema']}.{table_name}"
    
    try:
        # Get table history/metadata
        desc_df = spark.sql(f"DESCRIBE DETAIL {full_table_name}")
        details = desc_df.collect()[0]
        
        print(f"\n{table_name}:")
        print(f"  Location: {details['location']}")
        print(f"  Format: {details['format']}")
        print(f"  Last Modified: {details['lastModified']}")
        
        # Calculate time since last update
        last_modified = details['lastModified']
        time_diff = datetime.datetime.now() - last_modified.replace(tzinfo=None)
        hours_ago = time_diff.total_seconds() / 3600
        
        if hours_ago < 1:
            print(f"  ✓ Updated recently ({int(time_diff.total_seconds() / 60)} minutes ago)")
        elif hours_ago < 24:
            print(f"  ✓ Updated {int(hours_ago)} hours ago")
        else:
            print(f"  ⚠️  Last updated {int(hours_ago / 24)} days ago")
            
    except Exception as e:
        print(f"\n{table_name}: Unable to get details - {e}")

print("\n" + "=" * 60)
print("✓ Freshness check complete")

## Testing Incremental Ingestion

To verify that incremental ingestion is working:

### 1. Insert Test Data in MySQL
```sql
-- Run this in your MySQL database
INSERT INTO policy (policy_no, cust_id, policytype, pol_issue_date) 
VALUES ('TEST001', 'CUST999', 'Auto', '2026-03-31');
```

### 2. Wait for CDC Processing
* Gateway pipeline captures change from binlog (∼ few seconds)
* Ingestion pipeline applies change to table (∼1-2 minutes)

### 3. Verify in Databricks
```sql
SELECT * FROM main.claims_mysql.policy 
WHERE policy_no = 'TEST001';
```

### 4. Test Update
```sql
-- Update in MySQL
UPDATE policy SET premium = 1200.00 WHERE policy_no = 'TEST001';

-- Check in Databricks after ∼1-2 minutes
SELECT * FROM main.claims_mysql.policy WHERE policy_no = 'TEST001';
```

### 5. Test Delete
```sql
-- Delete in MySQL
DELETE FROM policy WHERE policy_no = 'TEST001';

-- Verify deletion in Databricks
-- For SCD Type 1: Record will be removed
-- For SCD Type 2: Record will be marked as historical
```

## ✓ Setup Complete!

You've successfully configured MySQL incremental ingestion using Lakeflow Connect.

### What's Running:
1. **Gateway Pipeline** (`claims_mysql_gateway`)
   * Continuously monitors MySQL binlog
   * Captures INSERT, UPDATE, DELETE operations
   * Stores CDC data in staging volumes

2. **Ingestion Pipeline** (`claims_mysql_ingestion`)
   * Applies changes to destination tables
   * Handles SCD Type 1 and Type 2
   * Can run on schedule or continuously

### Ingested Tables:
* `main.claims_mysql.policy` (SCD Type 1)
* `main.claims_mysql.claim` (SCD Type 1)
* `main.claims_mysql.customer` (SCD Type 2 - History Tracking)

### Monitoring:
* **Pipeline Status**: Check cells above for current state
* **Pipeline UI**: Navigate to Data Ingestion > Pipelines
* **Data Quality**: Run verification queries above
* **Alerting**: Configure email notifications in pipeline settings

### Best Practices:
* Keep gateway pipeline running continuously
* Monitor pipeline health and binlog lag
* Set up alerts for pipeline failures
* Regularly verify data freshness
* Test CDC with sample INSERT/UPDATE/DELETE operations

### Resources:
* [MySQL Lakeflow Connect Documentation](https://docs.databricks.com/ingestion/lakeflow-connect/mysql-pipeline.html)
* [SCD Type 2 History Tracking](https://docs.databricks.com/ingestion/lakeflow-connect/mysql-reference.html)
* [DLT Pipeline Monitoring](https://docs.databricks.com/workflows/delta-live-tables/delta-live-tables-event-log.html)